In [1]:
import os
import sys
from pathlib import Path

homework_folder = (
    Path.home()
    / "machine-learning-zoomcamp"
    / "cohorts"
    / "2026"
    / "homework"
    / "05-deployment"
)

os.chdir(homework_folder)

print("Python version:", sys.version)
print("Working folder:", Path.cwd())
print("Model file found:", Path("pipeline.bin").exists())

Python version: 3.11.15 (main, Aug  7 2026, 02:15:53) [Clang 22.1.3 ]
Working folder: /Users/yashbatra/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
Model file found: True


In [2]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.11.15 (main, Aug  7 2026, 02:15:53) [Clang 22.1.3 ]
Python executable: /Users/yashbatra/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment/.venv/bin/python


In [3]:
# Homework 5 — Deploying Machine Learning Models

## Question 1 — Environment check

I ran `uv --version` in Mac Terminal.

Output:
uv 0.13.5 (46776a95d 2026-10-21 aarch64-apple-darwin)



In [9]:
import sklearn
print(sklearn.__version__)

1.7.2


In [10]:
from pathlib import Path
import hashlib

model_path = Path("pipeline.bin")

checksum = hashlib.sha256(model_path.read_bytes()).hexdigest()

expected_checksum = (
    "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
)

assert checksum == expected_checksum, "Model checksum does not match!"

print("Correct model file verified.")

Correct model file verified.


In [11]:
# Question 3 — load the model

In [12]:
import pickle
from model import normalize_record

with open("pipeline.bin", "rb") as model_file:
    pipeline = pickle.load(model_file)

print("Model loaded successfully.")

Model loaded successfully.


In [13]:
lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

record = normalize_record(lead)

probabilities = pipeline.predict_proba([record])

print("Class order:", pipeline.classes_)
print("Probabilities:", probabilities)

conversion_probability = float(probabilities[0, 1])

print(f"Conversion probability: {conversion_probability:.3f}")

Class order: [0 1]
Probabilities: [[0.46700558 0.53299442]]
Conversion probability: 0.533


In [17]:
# Question 4 — serve the model

import json
from urllib.request import Request, urlopen

lead = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

request = Request(
    url="http://localhost:9696/predict",
    data=json.dumps(lead).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urlopen(request, timeout=10) as response:
    result = json.load(response)

print(result)
print(f"Conversion probability: {result['conversion_probability']:.3f}")

assert 0 <= result["conversion_probability"] <= 1
assert isinstance(result["conversion"], bool)

{'conversion_probability': 0.769799, 'conversion': True}
Conversion probability: 0.770


In [15]:
# Question 5 — look at the container configuration

In [18]:
from pathlib import Path

dockerfile = Path("Dockerfile").read_text()

for line in dockerfile.splitlines():
    if line.startswith("FROM "):
        print(line)

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


## Question 6 — Run the model with Docker

Answer: 0.770

Built the image:
docker build -t zoomcamp-model:2026-hw5 .

Started the container:
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5

Sent the Question 4 lead to POST /predict.

Result:
{'conversion_probability': 0.769799, 'conversion': True}

The Docker API returned the same prediction as the local API.